In [0]:
from pyspark.sql import functions as F


In [0]:
%run /Workspace/Users/sushantbhardwaj15@gmail.com/FMCG_Analytics/setup_utils_config/Utilities

In [0]:
def get_process_batch(table_name: str):
    def process_batch(batch_df, batch_id):
        if batch_df.isEmpty():
            return

        batch_df = (
            batch_df
            .withColumn("_source_file", F.col("_metadata.file_name"))
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_load_date", F.current_date())
        )
        (
            batch_df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(f"fmcg.bronze.{table_name}")
        )

        row_count = batch_df.count()

        write_audit(
            "bronze",
            table_name,
            row_count,
            "SUCCESS",
            f"batch_id={batch_id}"
        )

    return process_batch

In [0]:
def start_autoloader_stream(table_name: str, config: dict):
    print(f"Starting Autoloader stream for: {table_name.upper()}")
    stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", config["format"])
        .option("header", True)
        .option("cloudFiles.inferColumnTypes", True)  # recommended for bronze
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.schemaLocation", config["checkpoint"])
        .load(config["source"])
    )

    query = (
        stream.writeStream
        .foreachBatch(get_process_batch(table_name))
        .option("checkpointLocation", config["checkpoint"])
        .trigger(availableNow=True)
        .start()
    )

    return query

In [0]:
# ============================================================
# START ALL STREAMS
# ============================================================

def run_all_streams():
    queries = {}

    for table_name, config in TABLE_CONFIG.items():
        if(config["load_type"]=="incremental"):
            query = start_autoloader_stream(table_name, config)
            queries[table_name] = query

    # Wait for all streams to finish (trigger = availableNow)
    for table_name, query in queries.items():
        query.awaitTermination()
        print(f"✓ {table_name} stream completed")



run_all_streams()